In [1]:
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install cvxopt

  Using cached cvxopt-1.3.2-cp313-cp313-macosx_15_0_arm64.whl.metadata (1.3 kB)
Using cached cvxopt-1.3.2-cp313-cp313-macosx_15_0_arm64.whl (12.4 MB)


In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from cvxopt import matrix, solvers
from itertools import combinations
import random
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

### Load the data

In [3]:
file_path = "parquets"
df_dict = [];
for file in range(0,10):
  df = pd.read_parquet(f'{file_path}/Num_{file}.parquet')
  grouped = df.groupby(level='sheet')
  df_dict.append(grouped)
# every file is now found in the dict

### Determine dataset lenghts

In [4]:
lenghts_dataset = []
# checks the lengths per number
for i in range(10):
  lenghts_sing = []
  for f in range(1,101):
    study_df = df_dict[i]
    sample = study_df.get_group(f)
    lenghts_sing.append(len(sample))
  # appends to the overall list
  lenghts_dataset.append(lenghts_sing)

mean_lens = []
for i in range(10):
  mean_lens.append(np.mean(lenghts_dataset[i]))

mean = np.mean(mean_lens) # 55.25
median = np.median(mean_lens) # 54.87

### Interpolate the dataset to fixed length

In [5]:
def interpolate_data(points, target_length):
  #cumulative arc length
  diffs = np.diff(points, axis=0)
  segment_len = np.sqrt(np.sum(diffs**2, axis=1))
  cumlen = np.concatenate(([0], np.cumsum(segment_len)))
  total_len = cumlen[-1]
  # create target archs
  target_arc_len = np.linspace(0, total_len, target_length)
  # interpolate
  interpolated = np.zeros((target_length, points.shape[1]))
  for coord  in range(points.shape[1]):
    interpolated[:, coord] = np.interp(target_arc_len, cumlen, points[:, coord])
  # resample
  return interpolated

In [6]:
# interpolate each number
number_dict = {}
for number in range(10):
  target = 55 # global average
  key = str(number)
  interp_numbers = []
  for sample in range(1,101):
    study_df = df_dict[number]
    sample_df = study_df.get_group(sample)
    # call the interpolation
    intp_data1 = interpolate_data(sample_df.to_numpy(), target)
    interp_numbers.append(intp_data1)
  # add the dictionary
  number_dict[key] = interp_numbers

#### Determine models parameters

In [7]:
def svm_linear_fit(data, class_labels):
  N = data.shape[0] # number of samples
  LB = 0
  C = 1.0
  # let's define P q g H A b
  K = (data @ data.T)
  reg_factor = 1e-6 * np.trace(K) / N  # Adaptive regularization
  P_matrix = (np.outer(class_labels, class_labels) * K + reg_factor * np.eye(N))
  P = matrix(P_matrix.astype(float))
  q = matrix(-np.ones(N)) # correct
  G = matrix(np.vstack((-np.eye(N), np.eye(N)))) # correct
  h = matrix(np.hstack((np.zeros(N), C*np.ones(N)))) # correct
  A = matrix(class_labels.reshape(1, -1), tc='d') # correct
  b = matrix(0.0) # correct
  # solve the quadratic problem
  solvers.options['show_progress'] = False
  sol = solvers.qp(P, q, G, h, A, b)
  # get the lambdas
  lambdas = np.array(sol['x']).flatten()
  # get the support vectors
  tol = 1e-5
  supp_sv_idx = (lambdas > tol)
  sum_sp_idx = np.sum(supp_sv_idx)
  supp_vc = data[supp_sv_idx]
  supp_labels = class_labels[supp_sv_idx]
  # get the weights
  w = np.sum((lambdas * class_labels)[:,None]* data, axis=0)
  w0_value = supp_labels - np.dot(supp_vc,w)
  w0 = np.mean(w0_value)
  return [w, w0]

#### Train the model(s)

In [8]:
def train_svm_model(digit1, digit2):
    # prepare the data
    X = []
    y = []
    # digits to string
    dig1 = str(digit1)
    dig2 = str(digit2)
    # one digit
    for i in range(0, 80):  # 0 min 99 is max
        sequence = number_dict[dig1][i]
        X.append(sequence)
        y.append(-1)  # label -1 for digit1
    # other digits
    for i in range(0, 80):  # 0 min 99 is max
        sequence = number_dict[dig2][i]
        X.append(sequence)
        y.append(1)  # label +1 for digit2
    X_arr = np.array(X)
    y_arr = np.array(y)
    # flatten the data
    X_flat = X_arr.reshape(X_arr.shape[0], -1)
    # standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_flat)
    # get weights
    weights = svm_linear_fit(X_scaled, y_arr)
    return weights, scaler
# all combinatiosn
all_combi = list(combinations(range(10),2))
# save the models
models = {}
#for combination in combinations2:
for i, (d1, d2) in enumerate(all_combi):
  weights, scaler = train_svm_model(d1, d2)
  key = f'{d1}_{d2}'
  models[key] = {
      'w': weights[0],
      'w0': weights[1],
      'scaler': scaler,
      'digit1': d1,
      'digit2': d2
  }


#### Test on unseen data

In [9]:
def svm_predict(unknown_digit, model_datas, combis):
  # predict the digit using all pair-wise comparisions
  votes = {digit:0 for digit in range(10)}
  confidence_vals = {digit:0 for digit in range(10)}
  hydric_evalution = {}
  for d1,d2 in combis:
    key_open = f'{d1}_{d2}'
    model_info = model_datas[f'{d1}_{d2}']
    w = model_info['w']
    w0 = model_info['w0']
    scaler = model_info['scaler']
    # reshape the digit
    flat_digit = unknown_digit.reshape(1, -1)
    # standardize
    standardized_digit = scaler.transform(flat_digit)
    score = np.dot(standardized_digit, w) + w0
    confidence = abs(score)
    if score < 0: # - 1
      votes[d1] += 1
      confidence_vals[d1] += confidence
    else:
      votes[d2] += 1
      confidence_vals[d2] += confidence
  # compose a hybrid score
  for digit in range(10):
    hydric_evalution[digit] = votes[digit] * (1+confidence_vals[digit]/100)
  # determine the best prediction
  predicted_digit = max(hydric_evalution, key=hydric_evalution.get)
  return predicted_digit

In [10]:
rand_numbers = [str(i) for i in range(0,10)]
rand_indx = list(range(82,99))
true_label = []
true_t = 0
predicts = []
for t in range(1,100):
  # take 10 samples not found in training
  rand_digit = random.choice(rand_numbers)
  rand_index = random.choice(rand_indx)
  rand_sequence = number_dict[rand_digit][rand_index]
  true_label.append(int(rand_digit))
  # predict
  predicted_digit = svm_predict(rand_sequence, models, all_combi)
  predicts.append(predicted_digit)
  # determine the score
  if predicted_digit == int(rand_digit):
    true_t += 1
accuracy = true_t/len(true_label)
print(accuracy)
print("Confusion Matrix:")
cm = confusion_matrix(true_label, predicts)
print(cm)
print("\n")
print("Classification Report:")
print(classification_report(true_label, predicts))

0.9494949494949495
Confusion Matrix:
[[18  0  0  0  0  0  0  0  0  0]
 [ 0  7  0  0  0  0  0  1  0  0]
 [ 0  0  5  0  0  0  0  0  0  0]
 [ 0  0  0 12  0  0  0  0  0  0]
 [ 1  0  0  0  9  0  0  0  0  1]
 [ 0  0  0  0  0  6  0  0  1  0]
 [ 0  1  0  0  0  0 10  0  0  0]
 [ 0  0  0  0  0  0  0  7  0  0]
 [ 0  0  0  0  0  0  0  0  8  0]
 [ 0  0  0  0  0  0  0  0  0 12]]


Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.88      0.88      0.88         8
           2       1.00      1.00      1.00         5
           3       1.00      1.00      1.00        12
           4       1.00      0.82      0.90        11
           5       1.00      0.86      0.92         7
           6       1.00      0.91      0.95        11
           7       0.88      1.00      0.93         7
           8       0.89      1.00      0.94         8
           9       0.92      1.00      0.96        12

    accurac